In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # go from notebooks → Python
SRC_PATH = PROJECT_ROOT / "Python" / "src"

sys.path.append(str(SRC_PATH))

print("Added to path:", SRC_PATH)

# ============================================
# AIR WHEEL — MP4 DIRECTORY SCANNER
# NeuroMomentum Lab
# ============================================

from pathlib import Path
import json

ROOT = Path("/mnt/pdata_bak/Air_Wheel_Methods")

# --- animals and dates of interest ---
TARGETS = {
    "NML_04": "2026_01_24",
    "NML_05": "2026_01_14",
    "NML_06": "2026_01_16",
}

ROI_REF_ANIMAL = "NML_GC_01"
ROI_REF_DATE   = "2025_12_16"


def pick_video_file(folder: Path, prefix: str, allow_reduced=True):
    """
    Pick appropriate mp4 file based on prefix and reduction rules.
    """

    files = sorted(
        f for f in folder.glob(f"{prefix}*.mp4")
        if "reduced" not in f.name.lower()
    )

    if not files:
        return ""

    # pupil never uses reduced
    if prefix == "pupi":
        return str(files[0])

    # face/video prefer reduced
    if allow_reduced:
        reduced = [f for f in files if "_reduced" in f.name]
        if reduced:
            return str(reduced[0])

    return str(files[0])


def build_airwheel_dict():
    animals = []

    for animal, date in TARGETS.items():

        session_dir = ROOT / animal / date

        if not session_dir.exists():
            print(f"[WARNING] Missing folder: {session_dir}")
            continue

        info = {
            "ID": animal,
            "date": date,
            "session_dir": str(session_dir),
            "video": {
                "mp4": {
                    "face": "",
                    "pupil": "",
                    "paws": "",
                }
            },
            "roi_reference": {
                "animal": ROI_REF_ANIMAL,
                "date": ROI_REF_DATE,
            }
        }

        # ---------- detect videos ----------
        info["video"]["mp4"]["face"]  = pick_video_file(session_dir, "face")
        info["video"]["mp4"]["pupil"] = pick_video_file(session_dir, "pupi")
        info["video"]["mp4"]["paws"]  = pick_video_file(session_dir, "video")

        animals.append(info)

    return animals


# ============================================
# BUILD STRUCTURE
# ============================================

airwheel_data = build_airwheel_dict()

print("\n===== SUMMARY =====")
for a in airwheel_data:
    print(a["ID"], a["date"])
    print("  face :", a["video"]["mp4"]["face"])
    print("  pupil:", a["video"]["mp4"]["pupil"])
    print("  paws :", a["video"]["mp4"]["paws"])


Added to path: /home/nmldata2/ttpaw/Python/src

===== SUMMARY =====
NML_04 2026_01_24
  face : /mnt/pdata_bak/Air_Wheel_Methods/NML_04/2026_01_24/face_1440x1080_60_20260124_154254.mp4
  pupil: /mnt/pdata_bak/Air_Wheel_Methods/NML_04/2026_01_24/pupi_320x240_60_20260124_154254.mp4
  paws : /mnt/pdata_bak/Air_Wheel_Methods/NML_04/2026_01_24/video_20260124_154256.mp4
NML_05 2026_01_14
  face : /mnt/pdata_bak/Air_Wheel_Methods/NML_05/2026_01_14/face_1440x1080_60_20260114_164026.mp4
  pupil: /mnt/pdata_bak/Air_Wheel_Methods/NML_05/2026_01_14/pupi_320x240_60_20260114_164026.mp4
  paws : /mnt/pdata_bak/Air_Wheel_Methods/NML_05/2026_01_14/video_20260114_164026.mp4
NML_06 2026_01_16
  face : /mnt/pdata_bak/Air_Wheel_Methods/NML_06/2026_01_16/face_1440x1080_60_20260116_154145.mp4
  pupil: /mnt/pdata_bak/Air_Wheel_Methods/NML_06/2026_01_16/pupi_320x240_60_20260116_154145.mp4
  paws : /mnt/pdata_bak/Air_Wheel_Methods/NML_06/2026_01_16/video_20260116_154140.mp4


In [ ]:
import cv2
from pathlib import Path


def resize_videos_if_needed(data_list, scale=0.25, overwrite=False):
    """
    Resize face and paws videos to reduced versions.

    Behavior:
    - skips if reduced already exists (unless overwrite=True)
    - leaves pupil videos untouched
    - updates dictionary paths to point to reduced videos
    """

    targets = ['face', 'paws']

    for entry in data_list:
        mp4_files = entry['video']['mp4']

        for key in targets:
            input_path = mp4_files.get(key)
            if not input_path:
                continue
            # print(input_path)
            # continue
            input_path = Path(input_path)
            if not input_path.exists():
                print(f"[WARNING] Missing: {input_path}")
                continue

            # ---------- output path ----------
            output_path = input_path.with_name(
                input_path.stem + "_reduced" + input_path.suffix
            )

            # print(output_path)
            # continue

            # ---------- skip logic ----------
            if output_path.exists() and not overwrite:
                print(f"[SKIP] Reduced exists: {output_path.name}")
                
                # IMPORTANT: update dictionary to use reduced
                entry['video']['mp4'][key] = str(output_path)
                continue

            # ---------- open video ----------
            cap = cv2.VideoCapture(str(input_path))
            fps = cap.get(cv2.CAP_PROP_FPS)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            new_width = int(width * scale)
            new_height = int(height * scale)

            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(
                str(output_path),
                fourcc,
                fps,
                (new_width, new_height)
            )

            print(f"\n[RESIZE] {entry['ID']} {key}")
            print(f"  {width}x{height} → {new_width}x{new_height}")

            # ---------- frame loop ----------
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                resized_frame = cv2.resize(
                    frame,
                    (new_width, new_height),
                    interpolation=cv2.INTER_AREA
                )

                out.write(resized_frame)

            cap.release()
            out.release()

            # IMPORTANT: update dictionary to reduced
            entry['video']['mp4'][key] = str(output_path)

            print(f"[SAVED] {output_path.name}")

In [ ]:
resize_videos_if_needed(airwheel_data, scale=0.25)

In [ ]:
print(type(airwheel_data))
print(len(airwheel_data[0]))
print(airwheel_data[0].keys())

In [4]:
from utils.motion_analysis import run_comprehensive_motion_analysis

run_comprehensive_motion_analysis(
    airwheel_data,   # ✅ THIS is the key fix
    roi_exist=True,
    force_cpu=False
)

[INFO] GPU available: True
[INFO] Using GPU: True

--- PHASE 1: SELECT ROIs FOR ALL VIDEOS ---
[INFO] Local ROI missing. Using reference ROI.
[INFO] Using reference ROI: /mnt/pdata_bak/Air_Wheel_Methods/NML_GC_01/2025_12_16/face_1440x1080_60_20251216_165824_reduced_roi.json
[INFO] Local ROI missing. Using reference ROI.
[INFO] Using reference ROI: /mnt/pdata_bak/Air_Wheel_Methods/NML_GC_01/2025_12_16/video_20251216_165824_reduced_roi.json
[INFO] Local ROI missing. Using reference ROI.
[INFO] Using reference ROI: /mnt/pdata_bak/Air_Wheel_Methods/NML_GC_01/2025_12_16/pupi_320x240_60_20251216_165824_roi.json
[INFO] Local ROI missing. Using reference ROI.
[INFO] Using reference ROI: /mnt/pdata_bak/Air_Wheel_Methods/NML_GC_01/2025_12_16/face_1440x1080_60_20251216_165824_reduced_roi.json
[INFO] Local ROI missing. Using reference ROI.
[INFO] Using reference ROI: /mnt/pdata_bak/Air_Wheel_Methods/NML_GC_01/2025_12_16/video_20251216_165824_reduced_roi.json
[INFO] Local ROI missing. Using referen

Analyzing pupil: 100%|█████████▉| 76011/76013 [31:53<00:00, 39.73fr/s] 
